In [2]:
import requests
from bs4 import BeautifulSoup
from dataclasses import dataclass

In [3]:
@dataclass
class Player:
    player_name: str
    hero_name: str
    round_standing: list
    

In [ ]:
# manual capture, though it might be worth it to run it through the page tree tool
# now that we have https://fabtcg.com/organised-play/{year}
# would still need to manually get event dates

events = [
    {"calling-seattle-battle-hardened-silver-age-9-am-1214/": {"date":"2025-12-16"}},
    {"battle-hardened-hong-kong-2025/": {"date":"2025-12-21"}}
]

header = {"From":"emry.kinney@gmail.com","User-Agent":"Mozilla/5.0 (Windows NT 10.0; WOW64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/154.0.0.0 Safari/537.36 Vivaldi/8.2.4133.76"}

for event in events:
    event_url = list(event.keys())[0]
    page = []

    # Make a series, but break if request status code is 404
    round = range(1,20)

    standings_url = ["https://fabtcg.com/coverage/" + event_url + f"standings/{r}/" for r in round]

    for idx, url in enumerate(standings_url):
        response = requests.get(url, headers=header)

        if response.status_code == 200:
            soup = BeautifulSoup(response.text, "html")

            # No 404 when you go to round 1234
            # Searching for the container that says "sorry, nothing here"
            # If it doesn't find anything, then we're good
            # Otherwise, that's the end of swiss
            if not soup.find("div", {"class":"text-center py-10"}):
                page.append(response.text)
            else:
                event[event_url]["swiss"] = idx
                break
            
    print(event[event_url])

    #round 1 will be used to instantiate
    soup = BeautifulSoup(page[0], "html")
    table_records = soup.find_all("tr")
    
    player_list = [
        Player(
            player_name = record.find("span",{"class":"player-name"}).text,
            hero_name = record.find("span", {"class":"hero-name"}).text if record.find("span", {"class":"hero-name"}) else None,
            round_standing = [int(record.find("td", {"class":"rank"}).text) if record.find("td", {"class":"rank"}).text.isnumeric() else record.find("td", {"class":"rank"}).text]
            )
        for record in table_records[1:]
        ]
    
    for p in page[1:]:
        soup = BeautifulSoup(p, "html")
        table_records = soup.find_all("tr")

        for record in table_records[1:]:
            record_player = record.find("span",{"class":"player-name"}).text
            record_hero = record.find("span",{"class":"hero-name"}).text if record.find("span",{"class":"hero-name"}) else None
            current_standing = int(record.find("td", {"class":"rank"}).text) if record.find("td", {"class":"rank"}).text.isnumeric() else record.find("td", {"class":"rank"}).text

            for player in player_list:
                if player.player_name == record_player:
                    player.hero_name = record_hero
                    player.round_standing.append(current_standing)
    
    event[event_url]["players"] = player_list

{'date': '2025-12-16', 'swiss': 8}
{'date': '2025-12-21', 'swiss': 6}


In [59]:
event_df = event[event_url]["players"]